# Fine-tune East Frisian (Oostfräisk) TTS with Piper

This notebook fine-tunes a VITS model from [Piper](https://github.com/rhasspy/piper) on East Frisian Low Saxon data.

**Two training modes available:**

| Mode | Phonemizer | Pros | Cons |
|------|-----------|------|------|
| **`grapheme`** | None — raw characters | Simple, no preprocessing hack, keeps original spelling | Needs more data to learn pronunciation patterns |
| **`espeak`** | espeak-ng (German) | Leverages German pronunciation knowledge | Requires text preprocessing (ğ→ch, ó→oa, etc.) |

Set your preferred mode in the cell below.

**Requirements:** Google Colab with A100 GPU

In [1]:
# =============================================
# CONFIGURATION — set your preferred mode here
# =============================================

PHONEME_MODE = "grapheme"   # "grapheme" or "espeak"

# Training hyperparameters
BATCH_SIZE = 16
MAX_EPOCHS = 2000
QUALITY = "medium"          # "medium" (22050 Hz) or "high" (22050 Hz, larger model)

print(f"Mode: {PHONEME_MODE}")
print(f"Batch size: {BATCH_SIZE}, Max epochs: {MAX_EPOCHS}, Quality: {QUALITY}")

Mode: grapheme
Batch size: 16, Max epochs: 2000, Quality: medium


# 1. Install Dependencies

In [2]:
# System packages needed for building piper
!apt-get install -y espeak-ng build-essential cmake ninja-build

# Install Piper from the new OHF-Voice repo (includes training code)
!pip install -U pip
!git clone --depth 1 https://github.com/OHF-Voice/piper1-gpl.git /tmp/piper
!cd /tmp/piper && pip install -e '.[train]'
!cd /tmp/piper && bash build_monotonic_align.sh
!cd /tmp/piper && python setup.py build_ext --inplace

# Install piper-tts for inference testing
!pip install piper-tts

# HuggingFace Hub for checkpoint download
!pip install huggingface_hub

E: Could not open lock file /var/lib/dpkg/lock-frontend - open (13: Permission denied)
E: Unable to acquire the dpkg frontend lock (/var/lib/dpkg/lock-frontend), are you root?
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 10.5 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 22.0.2
    Uninstalling pip-22.0.2:
      Successfully uninstalled pip-22.0.2
Cloning into '/tmp/piper'...
remote: Enumerating objects: 117, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 117 (delta 11), reused 68 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (117/117), 4.62 MiB | 26.89 MiB/s, done.
Resolving deltas: 100% (11/11), done.
Obtaining file:///tmp/piper
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Using cached

# 2. Clone Repository & Setup

In [ ]:
!git clone https://VanModers:@github.com/VanModers/oostfraeisk_text_to_speech
%cd oostfraeisk_text_to_speech
!git pull

# 3. GPU Optimizations

In [3]:
import torch

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"TF32 enabled: {torch.backends.cuda.matmul.allow_tf32}")

GPU: NVIDIA RTX PRO 6000 Blackwell Workstation Edition
VRAM: 102.0 GB
TF32 enabled: True


# 4. Download Pretrained German Checkpoint

Fine-tune from the German **Thorsten** voice (medium quality = VITS @ 22050 Hz).

In [4]:
from huggingface_hub import hf_hub_download, list_repo_tree
import os

repo_id = "rhasspy/piper-checkpoints"

# Find the German Thorsten medium checkpoint
all_files = [
    f.rfilename for f in list_repo_tree(repo_id, repo_type="dataset")
    if hasattr(f, 'rfilename')
    and "thorsten" in f.rfilename
    and "medium" in f.rfilename
    and f.rfilename.endswith(".ckpt")
]
print("Available Thorsten medium checkpoints:")
for f in all_files:
    print(f"  {f}")

if all_files:
    pretrained_ckpt = hf_hub_download(
        repo_id=repo_id, filename=all_files[0], repo_type="dataset"
    )
else:
    pretrained_ckpt = hf_hub_download(
        repo_id=repo_id,
        filename="de/de_DE/thorsten/medium/epoch=3135-step=2702056.ckpt",
        repo_type="dataset"
    )

print(f"\nDownloaded: {pretrained_ckpt}")

/home/tidospecht/repos/oostfraeisk_text_to_speech/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Available Thorsten medium checkpoints:

Downloaded: /home/tidospecht/.cache/huggingface/hub/datasets--rhasspy--piper-checkpoints/snapshots/52588227e5a29f8c2afc6c31280e42119760ac86/de/de_DE/thorsten/medium/epoch=3135-step=2702056.ckpt


# 5. Preprocess Text (espeak mode only)

In the new Piper, preprocessing (phonemization + audio caching) is handled automatically during training. No separate preprocessing step is needed.

- **`grapheme`**: No text preprocessing. The trainer uses `--data.phoneme_type text` to treat raw characters as phonemes.
- **`espeak`**: We still need to convert East Frisian orthography → German-compatible forms in the CSV before training, since the trainer will use espeak-ng with a German voice.

In [5]:
# East Frisian → German phoneme mapping (only used in espeak mode)
custom_phoneme_map = {
    # Complex diphthongs / triphthongs (longest first)
    "öye": "öije",    # /œyə/ - göyen (gießen)
    "ööe": "ööö",    # extra-long ö
    "óóej": "ooai",  # /ɒ:ɛɪ/ - dóóejt (Tat)
    "âau": "aau",    # /a:ʊ/ - brâau
    "âaj": "aai",    # /a:ɪ/ - drâajen (drehen)
    "êer": "eer",    # /e:r/ - fêert (fährt)
    "êel": "eel",    # /e:l/ - fêelen (fühlen)

    # Circumflex (extra-long) vowels
    "ââ": "aa", "êê": "ee", "îî": "ii", "ôô": "oo", "ûû": "uu",
    "âa": "aa", "êe": "ee", "îi": "ii", "ôo": "oo", "ûu": "uu",

    # East Frisian specific vowels
    "óój": "oai", "óó": "oa", "ó": "oa",

    # ö-diphthongs
    "öy": "öi", "öej": "ööi", "öj": "öi",

    # ä-diphthongs
    "äie": "ääi", "äej": "ääi", "äj": "äi", "äi": "äi",

    # Basic diphthongs
    "ooj": "ooi", "oi": "oi", "ei": "ei",
    "aaj": "aai", "ai": "ai", "aau": "aau", "au": "au",

    # Consonants
    "ğ": "ch", "tj": "tsch",
}

_sorted_keys = sorted(custom_phoneme_map.keys(), key=len, reverse=True)

def preprocess_east_frisian(text: str) -> str:
    """Convert East Frisian orthography to German-compatible forms."""
    result = text
    for key in _sorted_keys:
        result = result.replace(key, custom_phoneme_map[key])
    return result

# Quick test
for t in ["Suldóót", "fandóóeğ", "drâajen"]:
    print(f"  {t} → {preprocess_east_frisian(t)}")

  Suldóót → Suldoat
  fandóóeğ → fandoaech
  drâajen → draaien


In [6]:
import shutil
from pathlib import Path

DATASET_DIR = Path("data/oostfraeisk")
metadata_path = DATASET_DIR / "metadata.csv"

if PHONEME_MODE == "espeak":
    # ── ESPEAK MODE ─────────────────────────────────────
    # Preprocess text: East Frisian → German-compatible
    backup_path = metadata_path.with_suffix('.csv.original')
    if not backup_path.exists():
        shutil.copy(metadata_path, backup_path)
        print(f"Backed up original to {backup_path}")
    else:
        shutil.copy(backup_path, metadata_path)
        print(f"Restored original from {backup_path}")

    with open(metadata_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    processed_lines = []
    for line in lines:
        parts = line.strip().split('|')
        if len(parts) >= 2:
            filename = parts[0]
            text = parts[1]
            processed = preprocess_east_frisian(text)
            if len(parts) == 3:
                processed_lines.append(f"{filename}|{processed}|{processed}\n")
            else:
                processed_lines.append(f"{filename}|{processed}\n")
        else:
            processed_lines.append(line)

    with open(metadata_path, 'w', encoding='utf-8') as f:
        f.writelines(processed_lines)

    print(f"Preprocessed {len(processed_lines)} lines for espeak mode")
    print(f"Example: {lines[0].strip()}")
    print(f"       → {processed_lines[0].strip()}")

elif PHONEME_MODE == "grapheme":
    print("Grapheme mode — no text preprocessing needed.")
    print("Original East Frisian text will be used as-is.")
    print("The trainer will handle character-to-ID mapping automatically.")
else:
    raise ValueError(f"Unknown PHONEME_MODE: {PHONEME_MODE}")

Grapheme mode — no text preprocessing needed.
Original East Frisian text will be used as-is.
The trainer will handle character-to-ID mapping automatically.


In [7]:
# Verify dataset is ready
import csv
from pathlib import Path

DATASET_DIR = Path("data/oostfraeisk")
wav_dir = DATASET_DIR / "wavs"
if not wav_dir.is_dir():
    wav_dir = DATASET_DIR / "wav"

metadata_path = DATASET_DIR / "metadata.csv"
num_lines = 0
missing = 0
with open(metadata_path, 'r', encoding='utf-8') as f:
    reader = csv.reader(f, delimiter='|')
    for row in reader:
        num_lines += 1
        filename = row[0]
        wav_path = wav_dir / f"{filename}.wav"
        if not wav_path.exists():
            wav_path = wav_dir / filename
        if not wav_path.exists():
            missing += 1
            if missing <= 5:
                print(f"WARNING: Missing {filename}")

print(f"\nDataset: {num_lines} utterances, {missing} missing audio files")
print(f"Audio dir: {wav_dir}")
print(f"Sample: ", end="")
with open(metadata_path, 'r', encoding='utf-8') as f:
    print(f.readline().strip())


Dataset: 1004 utterances, 0 missing audio files
Audio dir: data/oostfraeisk/wavs
Sample: sentence_0001|Ennerwor mank däi fräej mäiden tüsken 't Braukmer- un 't Auerkerland wor man dat lûud|Ennerwor mank däi fräej mäiden tüsken 't Braukmer- un 't Auerkerland wor man dat lûud


In [8]:
import csv
import json
import unicodedata
from pathlib import Path
from collections import Counter

DATASET_DIR = Path("data/oostfraeisk")
TRAINING_DIR = Path("piper_training")
TRAINING_DIR.mkdir(parents=True, exist_ok=True)

metadata_path = DATASET_DIR / "metadata.csv"

if PHONEME_MODE == "grapheme":
    # ── Build custom phoneme ID map for grapheme mode ───────
    # Piper does unicodedata.normalize("NFD", text) which decomposes
    # accented chars: ä→a+̈, ó→o+́, â→a+̂, ğ→g+̆
    # The default map only has basic lowercase + IPA.
    # We need to add: uppercase letters, combining diacritical marks,
    # and any other characters in our text.

    all_chars = Counter()
    with open(metadata_path, 'r', encoding='utf-8') as f:
        reader = csv.reader(f, delimiter='|')
        for row in reader:
            text = row[-1]
            # This is exactly what Piper does internally
            nfd_text = unicodedata.normalize("NFD", text)
            all_chars.update(nfd_text)

    print(f"Unique characters after NFD normalization ({len(all_chars)}):")
    for char, count in sorted(all_chars.items(), key=lambda x: -x[1]):
        name = unicodedata.name(char, f"U+{ord(char):04X}")
        print(f"  {repr(char):10s} (U+{ord(char):04X} {name}): {count}x")

    # Start from Piper's default map (IPA + basic Latin)
    from piper.phoneme_ids import DEFAULT_PHONEME_ID_MAP

    phoneme_id_map = dict(DEFAULT_PHONEME_ID_MAP)
    next_id = max(max(ids) for ids in phoneme_id_map.values()) + 1

    # Add any missing characters from our dataset
    missing = []
    for char in sorted(all_chars.keys()):
        if char not in phoneme_id_map:
            phoneme_id_map[char] = [next_id]
            missing.append((char, next_id))
            next_id += 1

    if missing:
        print(f"\nAdded {len(missing)} characters to phoneme ID map:")
        for char, pid in missing:
            name = unicodedata.name(char, f"U+{ord(char):04X}")
            print(f"  {repr(char):10s} → ID {pid}  ({name})")
    else:
        print("\nNo missing characters — default map covers everything.")

    # Save the custom map
    phonemes_path = TRAINING_DIR / "phoneme_id_map.json"
    with open(phonemes_path, "w", encoding="utf-8") as f:
        json.dump(phoneme_id_map, f, ensure_ascii=False, indent=2)

    num_symbols = max(max(ids) for ids in phoneme_id_map.values()) + 1
    print(f"\nSaved phoneme ID map to {phonemes_path}")
    print(f"Total entries: {len(phoneme_id_map)}, num_symbols needed: {num_symbols}")

else:
    print("espeak mode — using Piper's built-in phoneme ID map.")
    phonemes_path = None
    num_symbols = 256

Unique characters after NFD normalization (79):
  ' '        (U+0020 SPACE): 15225x
  'e'        (U+0065 LATIN SMALL LETTER E): 7669x
  'n'        (U+006E LATIN SMALL LETTER N): 7368x
  'a'        (U+0061 LATIN SMALL LETTER A): 6409x
  'o'        (U+006F LATIN SMALL LETTER O): 6298x
  'i'        (U+0069 LATIN SMALL LETTER I): 5691x
  't'        (U+0074 LATIN SMALL LETTER T): 5332x
  'r'        (U+0072 LATIN SMALL LETTER R): 4186x
  '̈'        (U+0308 COMBINING DIAERESIS): 4074x
  's'        (U+0073 LATIN SMALL LETTER S): 3884x
  'u'        (U+0075 LATIN SMALL LETTER U): 3741x
  'd'        (U+0064 LATIN SMALL LETTER D): 3634x
  'l'        (U+006C LATIN SMALL LETTER L): 2629x
  'k'        (U+006B LATIN SMALL LETTER K): 2529x
  'm'        (U+006D LATIN SMALL LETTER M): 2078x
  '́'        (U+0301 COMBINING ACUTE ACCENT): 1928x
  "'"        (U+0027 APOSTROPHE): 1705x
  'f'        (U+0066 LATIN SMALL LETTER F): 1596x
  'g'        (U+0067 LATIN SMALL LETTER G): 1560x
  'h'        (U+0068 LATI

# 6. Train the Model

Fine-tune from the pretrained German Thorsten checkpoint using the new `piper.train fit` CLI.

**Important:** Since the German checkpoint uses espeak phonemes and our grapheme model uses raw characters (different phoneme set), we use `--model.vocoder_warmstart_ckpt` for grapheme mode — this only copies the vocoder/acoustic parameters, not the phoneme embedding layer.

For espeak mode (where we've converted text to German-compatible forms), we can use `--ckpt_path` to load the full checkpoint.

**Tips:**
- Monitor `loss_g` in tensorboard — model is done when it plateaus
- espeak mode: ~1000 extra epochs usually sufficient
- grapheme mode: may need more epochs (~2000+)
- Adjust `--data.batch_size` if you run out of VRAM

In [9]:
from pathlib import Path

DATASET_DIR = Path("data/oostfraeisk")
TRAINING_DIR = Path("piper_training")
TRAINING_DIR.mkdir(parents=True, exist_ok=True)

config_path = TRAINING_DIR / "config.json"
cache_dir = TRAINING_DIR / "cache"

# Build the training command
base_cmd = [
    "python -m piper.train fit",
    f'--data.voice_name "oostfraeisk"',
    f"--data.csv_path {DATASET_DIR / 'metadata.csv'}",
    f"--data.audio_dir {DATASET_DIR / 'wavs'}",
    f"--model.sample_rate 22050",
    f"--data.espeak_voice de",
    f"--data.cache_dir {cache_dir}",
    f"--data.config_path {config_path}",
    f"--data.batch_size {BATCH_SIZE}",
    f"--data.validation_split 0.05",
    f"--data.num_test_examples 5",
    f"--trainer.max_epochs {MAX_EPOCHS}",
    "--trainer.accelerator gpu",
    "--trainer.devices 1",
    "--trainer.precision 32",
    f'--trainer.callbacks+=ModelCheckpoint --trainer.callbacks.every_n_epochs=5',
]

if PHONEME_MODE == "grapheme":
    base_cmd.append("--data.phoneme_type text")
    # Custom phoneme map with East Frisian characters + combining marks
    base_cmd.append(f"--data.phonemes_path {TRAINING_DIR / 'phoneme_id_map.json'}")
    base_cmd.append(f"--data.num_symbols {num_symbols}")
    # Use vocoder warmstart — different phoneme set than German checkpoint
    base_cmd.append(f'--model.vocoder_warmstart_ckpt "{pretrained_ckpt}"')
else:
    # Use full checkpoint — same espeak German phonemes
    base_cmd.append(f'--ckpt_path "{pretrained_ckpt}"')

cmd = " \\\n    ".join(base_cmd)
print("Training command:")
print(cmd)

Training command:
python -m piper.train fit \
    --data.voice_name "oostfraeisk" \
    --data.csv_path data/oostfraeisk/metadata.csv \
    --data.audio_dir data/oostfraeisk/wavs \
    --model.sample_rate 22050 \
    --data.espeak_voice de \
    --data.cache_dir piper_training/cache \
    --data.config_path piper_training/config.json \
    --data.batch_size 16 \
    --data.validation_split 0.05 \
    --data.num_test_examples 5 \
    --trainer.max_epochs 2000 \
    --trainer.accelerator gpu \
    --trainer.devices 1 \
    --trainer.precision 32 \
    --trainer.callbacks+=ModelCheckpoint --trainer.callbacks.every_n_epochs=5 \
    --data.phoneme_type text \
    --data.phonemes_path piper_training/phoneme_id_map.json \
    --data.num_symbols 196 \
    --model.vocoder_warmstart_ckpt "/home/tidospecht/.cache/huggingface/hub/datasets--rhasspy--piper-checkpoints/snapshots/52588227e5a29f8c2afc6c31280e42119760ac86/de/de_DE/thorsten/medium/epoch=3135-step=2702056.ckpt"


In [10]:
!python -m piper.train fit \
    --data.voice_name "oostfraeisk" \
    --data.csv_path data/oostfraeisk/metadata.csv \
    --data.audio_dir data/oostfraeisk/wavs \
    --model.sample_rate 22050 \
    --data.espeak_voice de \
    --data.cache_dir piper_training/cache \
    --data.config_path piper_training/config.json \
    --data.batch_size 16 \
    --data.validation_split 0.05 \
    --data.num_test_examples 5 \
    --trainer.max_epochs 2000 \
    --trainer.accelerator gpu \
    --trainer.devices 1 \
    --trainer.precision 32 \
    --trainer.callbacks+=ModelCheckpoint --trainer.callbacks.every_n_epochs=5 \
    --data.phoneme_type text \
    --data.phonemes_path piper_training/phoneme_id_map.json \
    --data.num_symbols 196 \
    --model.vocoder_warmstart_ckpt "/home/tidospecht/.cache/huggingface/hub/datasets--rhasspy--piper-checkpoints/snapshots/52588227e5a29f8c2afc6c31280e42119760ac86/de/de_DE/thorsten/medium/epoch=3135-step=2702056.ckpt"

/home/tidospecht/repos/oostfraeisk_text_to_speech/.venv/lib/python3.10/site-packages/lightning/fabric/utilities/seed.py:44: No seed found, seed set to 0
Seed set to 0
/home/tidospecht/repos/oostfraeisk_text_to_speech/.venv/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:piper.train.vits.dataset:Processing utterances...
INFO:piper.train.vits.dataset:Processed 1004 utterance(s)
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type     

In [ ]:
import subprocess, shlex

# Run the training command
result = subprocess.run(cmd, shell=True, check=False)
if result.returncode != 0:
    print(f"\nTraining exited with code {result.returncode}")
else:
    print("\nTraining complete!")

# 7. Find Best Checkpoint

In [11]:
import glob

# Check both possible checkpoint locations
ckpts = sorted(
    glob.glob("lightning_logs/version_*/checkpoints/*.ckpt") +
    glob.glob(str(TRAINING_DIR / "lightning_logs/version_*/checkpoints/*.ckpt"))
)

print("Available checkpoints:")
for ckpt in ckpts[-10:]:
    print(f"  {ckpt}")

if ckpts:
    best_ckpt = ckpts[-1]
    print(f"\nUsing latest: {best_ckpt}")
else:
    print("No checkpoints found!")

Available checkpoints:
  lightning_logs/version_0/checkpoints/epoch=1999-step=240000.ckpt

Using latest: lightning_logs/version_0/checkpoints/epoch=1999-step=240000.ckpt


# 8. Export to ONNX

Creates `model.onnx` for fast CPU inference using the new `piper.train.export_onnx` module. The config JSON written during training is copied alongside as `model.onnx.json`.

In [14]:
!pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 9.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [onnxscript]2 [onnxscript]


In [16]:
import os

!mkdir -p model_piper

# Patch export_onnx.py to use legacy ONNX exporter (PyTorch 2.6+ defaults
# to the dynamo-based exporter which can't trace VITS's data-dependent asserts)
!sed -i 's/verbose=False,/dynamo=False,\n        verbose=False,/' /tmp/piper/src/piper/train/export_onnx.py

!python -m piper.train.export_onnx \
    --checkpoint "{best_ckpt}" \
    --output-file model_piper/oostfraeisk.onnx

# Copy training config as ONNX config
!cp {str(TRAINING_DIR)}/config.json model_piper/oostfraeisk.onnx.json

onnx_path = "model_piper/oostfraeisk.onnx"
if os.path.exists(onnx_path):
    size_mb = os.path.getsize(onnx_path) / 1e6
    print(f"\nExported: {onnx_path} ({size_mb:.1f} MB)")
else:
    print("Export failed!")

/home/tidospecht/repos/oostfraeisk_text_to_speech/.venv/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/tmp/piper/src/piper/train/export_onnx.py:92: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/tmp/piper/src/piper/train/vits/attentions.py:235: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace 

# 9. Test Inference

In [17]:
from piper import PiperVoice
import wave

voice = PiperVoice.load("model_piper/oostfraeisk.onnx")
print(f"Model loaded! Sample rate: {voice.config.sample_rate}")
print(f"Phoneme type: {voice.config.phoneme_type}")

Model loaded! Sample rate: 22050
Phoneme type: text


In [19]:
test_sentences = [
    "Moin, woo gaajt 't dii?",
    "Denkent jii, dat ik disser sats gaud uutprooten dau?",
    "Hest duu däi süen fandóóeğ al säin?",
    "Däi oorsprungelk tóól fan däi Fräisen tüsken Laauwers un Wäiser was dat Olfräisk.",
]

for i, text in enumerate(test_sentences):
    # In espeak mode, preprocess; in grapheme mode, use as-is
    synth_text = preprocess_east_frisian(text) if PHONEME_MODE == "espeak" else text
    out_path = f"test_piper_{i}.wav"
    with wave.open(out_path, "w") as wav_file:
        wav_file.setnchannels(1)
        wav_file.setsampwidth(2)
        wav_file.setframerate(voice.config.sample_rate)
        voice.synthesize(synth_text, wav_file)
    print(f"[{i}] {text}")
    if PHONEME_MODE == "espeak":
        print(f"    → {synth_text}")
    print()

[0] Moin, woo gaajt 't dii?

[1] Denkent jii, dat ik disser sats gaud uutprooten dau?

[2] Hest duu däi süen fandóóeğ al säin?

[3] Däi oorsprungelk tóól fan däi Fräisen tüsken Laauwers un Wäiser was dat Olfräisk.



In [20]:
import IPython
IPython.display.Audio("test_piper_0.wav")

In [21]:
IPython.display.Audio("test_piper_1.wav")

In [ ]:
IPython.display.Audio("test_piper_2.wav")

In [ ]:
IPython.display.Audio("test_piper_3.wav")

# 10. Push to Git

In [ ]:
# Restore original metadata.csv if we modified it (espeak mode)
import shutil
from pathlib import Path

backup_path = Path("data/oostfraeisk/metadata.csv.original")
if backup_path.exists():
    shutil.copy(backup_path, "data/oostfraeisk/metadata.csv")
    print("Restored original metadata.csv")

In [ ]:
# Clean up training artifacts
!rm -rf piper_training/
!rm -rf lightning_logs/
print("Cleaned training artifacts")

In [ ]:
!git config --global user.email "programmingstudios227@gmail.com"
!git config --global user.name "Tido Specht"
!git add model_piper/
!git add -A
!git commit -m "Piper model treenäärt"
!git push

# 11. Deploy to HuggingFace Space

The HF Space needs:
- `model.onnx` + `model.onnx.json` — the exported model
- `app.py` — use `app_piper.py` from this repo
- `requirements.txt`: `piper-tts`, `gradio`

**Important:** If you trained in **grapheme mode**, set `PHONEME_MODE = "grapheme"` in `app_piper.py` — the East Frisian→German preprocessing will be skipped since the model understands native letters.

In [ ]:
!pip install huggingface_hub ipywidgets

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
%cd ..
!git clone https://huggingface.co/spaces/VanModers114/East_Frisian_TTS

In [ ]:
# Copy model + app to HF Space
!cp oostfraeisk_text_to_speech/model_piper/oostfraeisk.onnx East_Frisian_TTS/model.onnx
!cp oostfraeisk_text_to_speech/model_piper/oostfraeisk.onnx.json East_Frisian_TTS/model.onnx.json
!cp oostfraeisk_text_to_speech/app_piper.py East_Frisian_TTS/app.py

# Create requirements.txt
!echo "piper-tts" > East_Frisian_TTS/requirements.txt
!echo "gradio" >> East_Frisian_TTS/requirements.txt

In [ ]:
%cd East_Frisian_TTS
!git add -A
!git commit -m "Piper ONNX model"
!git push